# BanglaHalluEval Summarization Hallucination Pipeline
This notebook generates 3 hallucinated summaries per document using GPT-5.4.
Patterns: Intrinsic, Non-factual, Factual.


In [ ]:
!pip -q install openai python-dotenv

In [ ]:
import os
import csv
from pathlib import Path
from openai import OpenAI


## Configuration
Set your API key and paths below.

In [ ]:
os.environ['OPENAI_API_KEY'] = ''  # <- paste your key
MODEL = os.getenv('OPENAI_MODEL', 'gpt-5.4')
INPUT_CSV = '/content/summarization_pilot_20_random.csv'
OUTPUT_CSV = '/content/summarization_hallucinations_pilot_20.csv'
DOCUMENT_COLUMN = 'question'
SUMMARY_COLUMN = 'summary'
CHECKPOINT_EVERY = 50


In [ ]:
from google.colab import files

uploaded = files.upload()

if uploaded:
    INPUT_CSV = next(iter(uploaded.keys()))
    INPUT_CSV = f"/content/{INPUT_CSV}"


In [ ]:
def load_rows(csv_path):
    with open(csv_path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def write_rows(csv_path, rows, fieldnames):
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def build_prompt(template, document, right_summary):
    return template.replace('<Here is the test document>', document).replace(
        '<Here is the right summary of the test document>', right_summary
    )


def normalize_summary(text):
    cleaned = (text or '').strip()
    if not cleaned:
        return cleaned
    marker = '#Hallucinated Summary#'
    if marker in cleaned:
        cleaned = cleaned.split(marker, 1)[-1].strip()
    lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
    return lines[0] if lines else cleaned


In [ ]:
prompt_1 = (
    "I want you act as a hallucination summary generator. The answer should be given in BANGLA. Given a "
    "document and the right summary, your objective is to write a hallucinated summary that sounds plausible "
    "but is factually incorrect. You SHOULD write the hallucinated summary using the following method: "
    "You are trying to write a summary which is factual but some information "
    "cannot be directly inferred or entailed from the document.\n"
    "Example -\n"
    "#Question#: \"আমার স্ত্রীর বয়স ২২ বছর সে অন্তঃসত্ত্বা । প্রথমবার ডাক্তার গত 7 নভেম্বর 2022 তার প্রসাবের "
    "ডেট ছিল । কিন্তু গত সপ্তাহে ডাক্তার দেখালে ১৪ তারিখে ডেট দেয় । এখন কি করব আবার কি নতুন করে ডাক্তার "
    "দেখাবো নাকি ১৪ তারিখ পর্যন্ত অপেক্ষা করব\"\n"
    "#Right Summary#: \"স্ত্রীর বয়স ২২ । ৭ তারিখ প্রসাবের ডেট ছিল , কিন্তু ডাক্তার ১৪ তারিখে ডেট দেয় । কি "
    "করব\"\n"
    "#Hallucinated Summary#: \"মুখ্য সবিতার বয়স ২২ বছর, অন্তঃসত্ত্বা আছে। ডাক্তারের প্রথম দিক্ট ১১ জুলাই, "
    "গত সপ্তাহে নতুন ডিক্ট ১৪ তারিখে দেয়া হয়। এখন কি নতুন করে ডাক্তার দেখাব অথবা ১৪ পর্যন্ত অপেক্ষা করব?\"\n"
    "You should try your best to make the summary become hallucinated. #Hallucinated Summary# can only have "
    "about 5 more words than #Right Summary#.\n"
    "#Document#: <Here is the test document>\n"
    "#Right Summary#: <Here is the right summary of the test document>\n"
    "#Hallucinated Summary#: Generate"
 )

prompt_2 = (
    "I want you act as a hallucination summary generator. The answer should be given in BANGLA. Given a "
    "document and the right summary, your objective is to write a hallucinated summary that sounds plausible "
    "but is factually incorrect. You SHOULD write the hallucinated summary using the following method: "
    "You are trying to write a summary but there exist some non-factual and "
    "incorrect information. You can fabricate some information that does not exist in the provided document.\n"
    "Example -\n"
    "#Question#: আসসালামু আলাইকুম স্যার । আমার মেয়ের বয়স ৫ বছর । আলহামদুলিল্লাহ অসুখ বিসুখ খুবই কম । "
    "মোটামোটি পরিষ্কার পরিচ্ছন্ন থাকে সবসময় । কিন্তু ছোট বেলা থেকেই সর্দি সমস্যা । এখনও অনবরত পানির মত আবার "
    "কখনও পাকা সর্দি নাক দিয়ে ঝরতে থাকে যা বাচ্চার জন্য এমনকি আমাদের জন্যও খুবি অসস্থিকর । কি ব্যবহার করলে "
    "এ অবস্থা থেকে মুক্তি পাবে , অন্ততঃ সর্দি কমে যাবে জানালে উপকৃত হব । \"\n"
    "#Right Summary#: \"বয়স ৫ । ছোট থেকে পাকা সর্দি পানির মতো নাক থেকে ঝরতে থাকে । কি করণীয় ?\"\n"
    "#Hallucinated Summary#: \"আলহামদুলিল্লাহ আপনার ছেলে স্বাস্থ্যকর্ম ভাল, কিন্তু ধর্মটি এবং ঝাড়াগাড়ি দেয়। "
    "আপনি চান যেন সর্দি হ্রাস পায়, তাই মুশকিল কমানোর উপায় খুজছেন।\"\n"
    "You should try your best to make the summary become hallucinated. #Hallucinated Summary# can only have "
    "about 5 more words than #Right Summary#.\n"
    "#Document#: <Here is the test document>\n"
    "#Right Summary#: <Here is the right summary of the test document>\n"
    "#Hallucinated Summary#: Generate"
 )

prompt_3 = (
    "I want you act as a hallucination summary generator. The answer should be given in BANGLA. Given a "
    "document and the right summary, your objective is to write a hallucinated summary that sounds plausible "
    "but is factually incorrect. You SHOULD write the hallucinated summary using the following method: "
    "You are trying to write a summary but there is a factual contradiction "
    "between the summary and the document.\n"
    "Example -\n"
    "#Question#: \"কিছুক্ষণ বসে থাকলে পা তে মনে হয় পানি জমেছে ( আসলে না ) , কেমন যেন আনইজি লাগে । এটা অনেক দিন "
    "থেকে হচ্ছে । একটু হাটা চলা করলে ঠিক হয়ে যায় । কিন্তু কিছুক্ষণ পর আবার এমন হয় । আমার উচ্চতা অনুযায়ী ওজন "
    "ঠিক আছে । করনীয় কি ? ধন্যবাদ ।\"\n"
    "#Right Summary#: \"পায়ে পানি জমে ৷ হাটলে ঠিক হয়ে যায় । কি করনীয় ?\"\n"
    "#Hallucinated Summary#: \"পা জ্বলানোর অনুভূতি স্থায়ী হয় না, হাঁটা চলায় স্থায়ী হয়। উচ্চতা-ওজন অনুপাত "
    "নিয়মিত।  ডাক্তারের পরামর্শ কবুল করুন।\"\n"
    "You should try your best to make the summary become hallucinated. #Hallucinated Summary# can only have "
    "about 5 more words than #Right Summary#.\n"
    "#Document#: <Here is the test document>\n"
    "#Right Summary#: <Here is the right summary of the test document>\n"
    "#Hallucinated Summary#: Generate"
 )

PROMPT_MAP = {
    "Intrinsic": prompt_1,
    "Non-factual": prompt_2,
    "Factual": prompt_3,
}
